In [1]:
import pandas as pd
import numpy as np
import os, re, subprocess, pickle, gzip
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import accumulate
import warnings
warnings.filterwarnings('ignore')

In [2]:
FDRbench_path = "D:/entrapment_db"
working_folder = "C:/Users/Enrico/OneDrive - UGent/run-ionbot"
PXDs = [
    # 'PXD002057-entrap-closed',
    # 'PXD005833-entrap-closed',
    # 'PXD014258-entrap-closed',
    'PXD002057-entrapment',
    'PXD005833-entrapment',
    'PXD014258-entrapment',
]
SEARCHES = [ # do 1 search at the time for now
    # 'Pep-Canon',
    # 'Pep-Trembl',
    'Pep-Open',
]
DATASETS = pd.MultiIndex.from_product([PXDs,SEARCHES])
DATASETS

MultiIndex([('PXD002057-entrapment', 'Pep-Open'),
            ('PXD005833-entrapment', 'Pep-Open'),
            ('PXD014258-entrapment', 'Pep-Open')],
           )

In [3]:
FDRbench = pd.read_csv(
    os.path.join(FDRbench_path,f"FDRbench-{SEARCHES[0]}.csv.gz"),
    usecols=['sequence','proteins']
)
FDRbench.rename(columns={'sequence':'database_peptide','proteins':'entrap_proteins'}, inplace=True)
FDRbench.entrap_proteins = FDRbench.entrap_proteins.str.split(';')
FDRbench['isEntrapment'] = FDRbench.entrap_proteins.apply(lambda x: np.any([_.endswith('_p_target') for _ in x]))
FDRbench

,database_peptide,entrap_proteins,isEntrapment
0,CHFFKKTSLLK,[op|IP_3774812|TX=9606],False
1,CFSHFKTKLLK,[op|IP_3774812|TX=9606_p_target],True
2,ISVSRQTLFEDSFQQIMNMK,"[op|A0A024R711|TX=9606, op|O00308-2|TX=9606, o...",False
3,IMTFVIQSSFDQMLRENSQK,"[op|A0A024R711|TX=9606, op|O00308-2|TX=9606, o...",True
4,TEASETRKWTGTQFGQWDTAGFENEDQK,"[op|Q1ED39|TX=9606, op|H3BNU8|TX=9606, op|H3BP...",False
...,...,...,...
14391925,PSLQELKPRTR,[op|IP_292992|TX=9606_p_target],True
14391926,MQINAEEVVVGDLVEVKGGDR,"[op|A0A0S2Z3W6|TX=9606, op|IP_679829|TX=9606, ...",False
14391927,ELEAGINVQVGEVMGDDVKVR,"[op|A0A0S2Z3W6|TX=9606, op|IP_679829|TX=9606, ...",True
14391928,AYNHIASFIDNYLHIFIYIFI,[op|IP_680968|TX=9606],False


---------

In [4]:
folders = {search:[] for search in SEARCHES}
for search in SEARCHES:   
    for dataset_name in PXDs:
        for fld in os.scandir(os.path.join(working_folder, dataset_name, f"{dataset_name}-{search}")):
            if not fld.name.startswith('.') and os.path.isdir(fld.path): 
                folders[search].append((dataset_name,fld))
folders

{'Pep-Open': [('PXD002057-entrapment',
   <DirEntry '130327_o2_01_hu_C1_2hr-pep-open'>),
  ('PXD002057-entrapment', <DirEntry '130327_o2_02_hu_P1_2hr-pep-open'>),
  ('PXD002057-entrapment', <DirEntry '130327_o2_03_hu_C2_2hr-pep-open'>),
  ('PXD002057-entrapment', <DirEntry '130327_o2_04_hu_P2_2hr-pep-open'>),
  ('PXD002057-entrapment', <DirEntry '130327_o2_05_hu_C3_2hr-pep-open'>),
  ('PXD002057-entrapment', <DirEntry '130327_o2_06_hu_P3_2hr-pep-open'>),
  ('PXD005833-entrapment', <DirEntry 'AM10-pep-open'>),
  ('PXD005833-entrapment', <DirEntry 'AM11-pep-open'>),
  ('PXD005833-entrapment', <DirEntry 'AM12-pep-open'>),
  ('PXD005833-entrapment', <DirEntry 'AM13-pep-open'>),
  ('PXD005833-entrapment', <DirEntry 'AM14-pep-open'>),
  ('PXD005833-entrapment', <DirEntry 'AM15-pep-open'>),
  ('PXD005833-entrapment', <DirEntry 'AM16-pep-open'>),
  ('PXD005833-entrapment', <DirEntry 'AM17-pep-open'>),
  ('PXD005833-entrapment', <DirEntry 'AM18-pep-open'>),
  ('PXD005833-entrapment', <DirEntry 

In [5]:
def get_FDP(df):
    n_entrap = np.array([_ for _ in accumulate(df.isEntrapment.apply(int))])
    n_ids    = np.array(range(1,len(df)+1))
    FDP_rate = n_entrap / n_ids
    return FDP_rate

In [6]:
def get_maxrank_for_filtering(sample_fld_path,FDRbench_):
    data = pd.read_csv(os.path.join(sample_fld_path,'ionbot.first.csv'))    
    data = data.merge(FDRbench_, on='database_peptide', how='left')
    
    data2 = data[~data.proteins.str.contains('CONTAMINANT')].copy(deep=True)
    data2['isEntrapment'] = data2['isEntrapment'].fillna(True)
    data2.sort_values('psm_score', ascending=False, inplace=True)
    data2.reset_index(drop=True, inplace=True)

    data2['FDP'] = get_FDP(data2)
    
    return data2[(data2.FDP<.01)&(~data2.isEntrapment)].copy()

In [7]:
maxranks = {}
for dataset_name,sample_fld in folders['Pep-Open']:
    file = re.sub(r'-pep-open$','',sample_fld.name)
    psms_w_fdp = get_maxrank_for_filtering(sample_fld.path, FDRbench)
    try:
        maxranks[dataset_name][file] = len(psms_w_fdp)
    except:
        maxranks[dataset_name] = {}
        maxranks[dataset_name][file] = len(psms_w_fdp)

maxranks

{'PXD002057-entrapment': {'130327_o2_01_hu_C1_2hr': 5139,
  '130327_o2_02_hu_P1_2hr': 9149,
  '130327_o2_03_hu_C2_2hr': 3806,
  '130327_o2_04_hu_P2_2hr': 7873,
  '130327_o2_05_hu_C3_2hr': 4111,
  '130327_o2_06_hu_P3_2hr': 8070},
 'PXD005833-entrapment': {'AM10': 9727,
  'AM11': 10043,
  'AM12': 11175,
  'AM13': 10318,
  'AM14': 10754,
  'AM15': 8983,
  'AM16': 12140,
  'AM17': 8227,
  'AM18': 8935,
  'AM19': 11229,
  'AM20': 8537,
  'AM21': 7736,
  'AM7': 7380,
  'AM8': 8484,
  'AM9': 10303},
 'PXD014258-entrapment': {'Sample-BT474': 35209,
  'Sample-MCF': 35622,
  'SampleHela': 27079}}

In [8]:
with gzip.open(f'maxranks-for-FDP-filtering-{SEARCHES[0]}.gz','wb') as OUT:
    pickle.dump(maxranks,OUT)

In [9]:
# # to read the pickle
# with gzip.open('maxranks-for-FDP-filtering.gz','rb') as IN:
#     maxranks2 = pickle.load(IN)

-----

In [10]:
from utility_functions import *

In [11]:
PXDs = [
    'PXD002057.v0.11.4',
    'PXD005833.v0.11.4',
    'PXD014258.v0.11.4',
]
SEARCHES = [
    'openprot',
]

In [12]:
folders = {search:[] for search in SEARCHES}
for search in SEARCHES:   
    for dataset_name in PXDs:
        for fld in os.scandir(os.path.join(working_folder, dataset_name, f"{dataset_name}-{search}")):
            if not fld.name.startswith('.') and os.path.isdir(fld.path): 
                folders[search].append((dataset_name,fld))

In [13]:
## PSM per exp using current filtering methods
psm_counts = {}
for dataset_name,sample_fld in folders['openprot']:
    tmp = import_pep_IDs(
        os.path.join(sample_fld,"group-walk-output.csv"),
        filtering='groupwalk'
    )
    try:
        psm_counts[dataset_name][sample_fld.name] = len(tmp)
    except:
        psm_counts[dataset_name] = {}
        psm_counts[dataset_name][sample_fld.name] = len(tmp)
del tmp
psm_counts

{'PXD002057.v0.11.4': {'130327_o2_01_hu_C1_2hr-openprot': 6622,
  '130327_o2_02_hu_P1_2hr-openprot': 10890,
  '130327_o2_03_hu_C2_2hr-openprot': 4779,
  '130327_o2_04_hu_P2_2hr-openprot': 9602,
  '130327_o2_05_hu_C3_2hr-openprot': 5154,
  '130327_o2_06_hu_P3_2hr-openprot': 9744},
 'PXD005833.v0.11.4': {'AM10-openprot': 10450,
  'AM11-openprot': 10852,
  'AM12-openprot': 12004,
  'AM13-openprot': 11445,
  'AM14-openprot': 11587,
  'AM15-openprot': 9702,
  'AM16-openprot': 13066,
  'AM17-openprot': 9112,
  'AM18-openprot': 10081,
  'AM19-openprot': 11990,
  'AM20-openprot': 9414,
  'AM21-openprot': 8517,
  'AM7-openprot': 8136,
  'AM8-openprot': 9441,
  'AM9-openprot': 11144},
 'PXD014258.v0.11.4': {'ESC-HF-Sample-BT474-openprot': 47060,
  'ESC-HF-Sample-MCF-openprot': 46364,
  'ESC-HF-SampleHela-openprot': 36007}}